In [2]:
import pandas as pd

In [3]:
df = pd.read_parquet("data/processed/interactions.parquet")

In [4]:
print(f"[interactions] rows={len(df):,}, cols={df.shape[1]}")
print("\n-- dtypes --")
print(df.dtypes)

[interactions] rows=1,177,175, cols=25

-- dtypes --
ClientID                                  Int64
ProductID                                 Int64
SaleTransactionDate         datetime64[ns, UTC]
txn_date                    datetime64[ns, UTC]
StoreID                                   Int64
StoreCountry                     string[python]
Quantity                                  Int64
SalesNetAmountEuro                      float64
ClientSegment                    string[python]
ClientCountry                    string[python]
ClientOptINEmail                          Int64
ClientOptINPhone                          Int64
ClientGender                             object
Age                                     float64
Category                         string[python]
FamilyLevel1                     string[python]
FamilyLevel2                     string[python]
Universe                         string[python]
txn_year                                  int32
txn_month                          

In [5]:
# Basic nulls
print("\n-- top null % --")
print((df.isna().mean().sort_values(ascending=False).head(12) * 100).round(2))


-- top null % --
Age                        59.56
ClientGender                4.27
AvailableInStoreCountry     0.00
is_weekend                  0.00
txn_week                    0.00
txn_dow                     0.00
txn_month                   0.00
txn_year                    0.00
Universe                    0.00
FamilyLevel2                0.00
FamilyLevel1                0.00
Category                    0.00
dtype: float64


In [6]:
# Date sanity
print("\n-- date range --")
print(df["txn_date"].min(), "→", df["txn_date"].max())


-- date range --
2023-01-01 00:00:00+00:00 → 2025-02-15 00:00:00+00:00


In [7]:
for c in ["txn_year", "txn_month", "txn_dow", "txn_week", "is_weekend"]:
    if c in df.columns:
        print(f"{c}: min={df[c].min()} max={df[c].max()} unique={df[c].nunique()}")

txn_year: min=2023 max=2025 unique=3
txn_month: min=1 max=12 unique=12
txn_dow: min=0 max=6 unique=7
txn_week: min=1 max=52 unique=52
is_weekend: min=0 max=1 unique=2


In [8]:
# Availability flags (share of rows with availability=1)
for c in ["AvailableInStoreCountry", "AvailableInClientCountry"]:
    if c in df.columns:
        print(f"\n{c}: mean={df[c].mean():.3f} (dtype={df[c].dtype}) value_counts:\n{df[c].value_counts(dropna=False)}")



AvailableInStoreCountry: mean=0.471 (dtype=int8) value_counts:
AvailableInStoreCountry
0    622250
1    554925
Name: count, dtype: int64

AvailableInClientCountry: mean=0.465 (dtype=int8) value_counts:
AvailableInClientCountry
0    629555
1    547620
Name: count, dtype: int64


In [9]:
# Key coverage
print("\n-- key coverage --")
print("Unique clients:", df["ClientID"].nunique())
print("Unique products:", df["ProductID"].nunique())
print("Unique stores:", df["StoreID"].nunique())


-- key coverage --
Unique clients: 304929
Unique products: 29730
Unique stores: 151


In [10]:
# Basic validity checks
neg_qty = (df["Quantity"] < 0).sum() if "Quantity" in df.columns else 0
neg_amt = (df["SalesNetAmountEuro"] < 0).sum() if "SalesNetAmountEuro" in df.columns else 0
print(f"\nnegatives -> qty:{neg_qty}, amount:{neg_amt}")


negatives -> qty:0, amount:0


In [11]:
assert df["txn_date"].notna().mean() > 0.95, "Too many missing txn_date"
if "AvailableInStoreCountry" in df.columns:
    assert set(df["AvailableInStoreCountry"].unique()) <= {0,1}, "Availability must be 0/1"
if "AvailableInClientCountry" in df.columns:
    assert set(df["AvailableInClientCountry"].unique()) <= {0,1}, "Availability must be 0/1"
print("\n[OK] quick checks passed")


[OK] quick checks passed


In [12]:
# Check for duplicates
duplicate_rows = df[df.duplicated()]
print(f"\nNumber of duplicate rows: {len(duplicate_rows)}")
if not duplicate_rows.empty:
    print("\nSample duplicate rows:")
    print(duplicate_rows.head())


Number of duplicate rows: 5121

Sample duplicate rows:
                 ClientID            ProductID       SaleTransactionDate  \
29463  784139077380372687  9199855114467326563 2024-12-17 00:00:00+00:00   
33085  784139077380372687  3570677365406858878 2023-07-31 00:00:00+00:00   
40639  784139077380372687  6437654160590574503 2024-12-16 00:00:00+00:00   
41468  784139077380372687   600191190789987226 2024-12-18 00:00:00+00:00   
43773  784139077380372687  4141508142946398531 2023-07-31 00:00:00+00:00   

                       txn_date              StoreID StoreCountry  Quantity  \
29463 2024-12-17 00:00:00+00:00  5060113386821017236          USA         1   
33085 2023-07-31 00:00:00+00:00  5060113386821017236          USA         2   
40639 2024-12-16 00:00:00+00:00  5060113386821017236          USA         1   
41468 2024-12-18 00:00:00+00:00  5060113386821017236          USA         1   
43773 2023-07-31 00:00:00+00:00  5060113386821017236          USA         2   

       Sales